In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
try:
    from ipywidgets import interact, IntSlider
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    from ipywidgets import interact, IntSlider

# Ensure paths are correct for your local structure
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

from src.densoinn import DenSOINN
from baseframework.datahandler import load_data

# ==========================================
# 1. VISUALIZATION HELPERS
# ==========================================

def _draw_nodes_and_edges(ax, nodes, edges, node_colors, default_color='black',
                           highlight_nodes=None, highlight_color=None, highlight_size=180):
    """
    Shared low-level renderer: draws topological edges then neural nodes on
    a given axis, colored per node_colors = {node_id: color}.
    highlight_nodes (optional) get drawn larger/with a distinct marker, e.g.
    to mark density-peak cluster centers.
    """
    drawn_edges = set()
    for src_id, targets in edges.items():
        if src_id not in nodes:
            continue
        c1 = nodes[src_id]['centroid']
        color = node_colors.get(src_id, default_color)
        for tgt_id in targets:
            if tgt_id not in nodes:
                continue
            edge_tuple = tuple(sorted((src_id, tgt_id)))
            if edge_tuple not in drawn_edges:
                c2 = nodes[tgt_id]['centroid']
                ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color=color,
                        linewidth=1.6, alpha=0.75, zorder=2)
                drawn_edges.add(edge_tuple)

    highlight_nodes = highlight_nodes or set()
    for node_id, node_data in nodes.items():
        centroid = node_data['centroid']
        color = node_colors.get(node_id, default_color)
        if node_id in highlight_nodes:
            ax.scatter(centroid[0], centroid[1], color=highlight_color or color,
                       marker='*', edgecolor='black', linewidth=0.8,
                       s=highlight_size, zorder=4)
        else:
            ax.scatter(centroid[0], centroid[1], color=color, marker='X',
                       edgecolor='black', linewidth=0.5, s=70, zorder=3)


def _style_axis(ax, title):
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_aspect('equal', adjustable='datalim')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])


def plot_densoinn_stages(X_raw, nodes, edges, components, centers, macro_clusters,
                          batch_name="Current"):
    """
    Renders the four key stages of one DenSOINN offline-clustering pass,
    side by side:
      1. Raw network    - raw data + nodes/edges, no clustering info yet
      2. Components     - nodes colored by weakly-connected component
      3. Density peaks   - same components, with chosen cluster centers starred
      4. Macro-clusters  - final labels after cascading, the algorithm's output
    """
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    cmap = plt.get_cmap('tab20')

    # ---- Panel 1: raw network, no clustering yet ----
    ax = axes[0]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.6, zorder=1)
    _draw_nodes_and_edges(ax, nodes, edges, node_colors={}, default_color='dimgrey')
    _style_axis(ax, f"Network (raw data + nodes)\n{len(nodes)} nodes")

    # ---- Panel 2: connected components ----
    ax = axes[1]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.4, zorder=1)
    comp_color = {}
    for i, comp in enumerate(components):
        c = cmap(i % 20)
        for nid in comp:
            comp_color[nid] = c
    _draw_nodes_and_edges(ax, nodes, edges, node_colors=comp_color)
    _style_axis(ax, f"Connected components\n{len(components)} component(s)")

    # ---- Panel 3: density peaks / centers per component ----
    ax = axes[2]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.4, zorder=1)
    _draw_nodes_and_edges(ax, nodes, edges, node_colors=comp_color,
                          highlight_nodes=centers, highlight_color='red', highlight_size=220)
    _style_axis(ax, f"Density-peak centers\n{len(centers)} center(s) found")

    # ---- Panel 4: final macro-clusters ----
    ax = axes[3]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.4, zorder=1)
    unique_mcs = sorted(set(macro_clusters.values()))
    mc_color_map = {mc_id: cmap(i % 20) for i, mc_id in enumerate(unique_mcs)}
    macro_colors = {nid: mc_color_map.get(lbl, 'grey') for nid, lbl in macro_clusters.items()}
    _draw_nodes_and_edges(ax, nodes, edges, node_colors=macro_colors)
    _style_axis(ax, f"Macro-clusters (final output)\n{len(unique_mcs)} cluster(s)")

    # Shared legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Raw Data', markerfacecolor='gainsboro', markersize=8),
        Line2D([0], [0], marker='X', color='w', label='Neural Node', markerfacecolor=cmap(0), markeredgecolor='black', markersize=9),
        Line2D([0], [0], marker='*', color='w', label='Cluster Center', markerfacecolor='red', markeredgecolor='black', markersize=13),
        Line2D([0], [0], color=cmap(0), lw=1.6, label='Topological Edge'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(f"DenSOINN Pipeline - {batch_name}", fontsize=15, fontweight='bold', y=1.03)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()


# ==========================================
# 2. SETUP PARAMETERS & LOAD REAL DATA
# ==========================================
dataset_name = 'spiral'
batch_size = 300

# DenSOINN Hyperparameters (Tuned for complex9 based on your earlier discussions)
gamma = 0.3           # Edge decay speed
stream_speed = 120     # Denoising frequency
epsilon = 0.3          # Denoising intensity
max_nodes = 1000       # Network capacity limit
alpha = 20         # Offline peak sensitivity multiplier
intrinsic_dim = 2      # Dimensionality for density estimation

print(f"Loading '{dataset_name}' data...")
# DenSOINN's self-adaptive metric handles unnormalized data natively, but we follow the runner's scaling default
X, y = load_data(dataset_name, scale=True)

densoinn = DenSOINN(
    gamma=gamma,
    stream_speed=stream_speed,
    epsilon=epsilon,
    max_nodes=max_nodes,
    alpha=alpha,
    intrinsic_dim=intrinsic_dim
)

num_batches = len(X) // batch_size
stream_history = []
all_X = []

print(f"Processing {num_batches} batches...")

# ==========================================
# 3. RUN ALGORITHM & CAPTURE STATE
# ==========================================
for i in range(num_batches + 1):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(X))
    batch = X[start_idx:end_idx]

    if len(batch) == 0:
        break

    timestamps = np.arange(start_idx, end_idx)

    # Run Layer 1 (Online Phase)
    densoinn.learn(batch, timestamps)
    all_X.extend(batch)

    # --- STATE EXTRACTION ---
    # 1. Deep copy the Neural Nodes
    current_nodes = {}
    for n_id, node in densoinn.online_phase.network.nodes.items():
        current_nodes[n_id] = {'centroid': node.w.copy()}

    # 2. Extract the Topological Edges (Weakly connected)
    current_edges = {}
    for src, tgts in densoinn.online_phase.network.edges.items():
        current_edges[src] = list(tgts.keys())

    # 3. Trigger Layer 2 (Offline Phase) with full stage breakdown:
    # components, density-peak centers, and final macro-cluster labels.
    summaries = densoinn.online_phase.get_summary()
    stages = densoinn.offline_phase.predict_summaries_with_stages(summaries)

    # Save the frame
    stream_history.append({
        'batch_name': f'Batch {i} (t={timestamps[-1]})',
        'X_raw': np.array(all_X),
        'nodes': current_nodes,
        'edges': current_edges,
        'components': stages['components'],
        'centers': stages['centers'],
        'macro_clusters': stages['labels'],
    })

print("Stream processing complete! Ready for visualization.")

# ==========================================
# 4. INTERACTIVE RENDERER
# ==========================================
def view_real_stream(batch_index):
    state = stream_history[batch_index]
    plot_densoinn_stages(
        X_raw=state['X_raw'],
        nodes=state['nodes'],
        edges=state['edges'],
        components=state['components'],
        centers=state['centers'],
        macro_clusters=state['macro_clusters'],
        batch_name=state['batch_name']
    )

# Launch the interactive slider
interact(view_real_stream, batch_index=IntSlider(min=0, max=len(stream_history)-1, step=1, value=0, description='Time Step:'))

Loading 'spiral' data...
Processing 20 batches...
Stream processing complete! Ready for visualization.


interactive(children=(IntSlider(value=0, description='Time Step:', max=20), Output()), _dom_classes=('widget-i…

<function __main__.view_real_stream(batch_index)>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

# ==========================================
# 1. Generate Minimal "Island" Data
# ==========================================
np.random.seed(42)
# Create a component with very few nodes for maximum clarity
peak1 = np.random.normal(loc=[3, 3], scale=0.6, size=(10, 2))
peak2 = np.random.normal(loc=[7, 6], scale=0.7, size=(12, 2))
bridge = np.random.normal(loc=[5, 4.5], scale=0.5, size=(4, 2))
X = np.vstack([peak1, peak2, bridge])

# Simulate Local Density (p_i) using Gaussian KDE
kde = gaussian_kde(X.T, bw_method=0.4)
densities = kde(X.T) * 100 # Scale up for Z-axis visibility

# Create a smooth grid to draw the mountain surface
x_grid = np.linspace(1, 9, 100)
y_grid = np.linspace(1, 8, 100)
xx, yy = np.meshgrid(x_grid, y_grid)
positions = np.vstack([xx.ravel(), yy.ravel()])
zz = np.reshape(kde(positions).T, xx.shape) * 100

# ==========================================
# 2. Simulate Step 2 Math (Find q_i and Peaks)
# ==========================================
q = {}
h = np.zeros(len(X))
sorted_indices = np.argsort(densities)[::-1]

# Global peak handling
global_peak = sorted_indices[0]
q[global_peak] = global_peak 
h[global_peak] = np.max(np.linalg.norm(X - X[global_peak], axis=1)) * densities[global_peak]

# Find nearest higher-density node (q_i) and calculate Peak Score (h_i)
for k in range(1, len(sorted_indices)):
    i = sorted_indices[k]
    candidates = sorted_indices[:k] 
    dists = np.linalg.norm(X[candidates] - X[i], axis=1)
    min_idx = np.argmin(dists)
    
    q[i] = candidates[min_idx]
    h[i] = densities[i] * dists[min_idx]

# Apply the Alpha Threshold
h_mean = np.mean(h)
alpha = 2.0
centers = [i for i in range(len(X)) if h[i] >= alpha * h_mean]

# ==========================================
# 3. Plotting the 3D Academic Figure
# ==========================================
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# 1. Draw the Mountain Surface
ax.plot_surface(xx, yy, zz, cmap='Blues', alpha=0.3, edgecolor='none', zorder=0)

# 2. Draw the Standard Neural Nodes
non_centers = [i for i in range(len(X)) if i not in centers]
ax.scatter(X[non_centers, 0], X[non_centers, 1], densities[non_centers], 
           c='black', s=50, depthshade=True, zorder=2)

# 3. Draw the "Uphill" Paths (Distance to q_i)
for i in range(len(X)):
    if i != global_peak and i not in centers:
        target = q[i]
        ax.plot([X[i,0], X[target,0]], 
                [X[i,1], X[target,1]], 
                [densities[i], densities[target]], 
                color='dimgray', linestyle='--', linewidth=1.5, alpha=0.8)

# 4. Highlight the Peaks (Red Nodes)
ax.scatter(X[centers, 0], X[centers, 1], densities[centers], 
           c='red', marker='o', s=120, edgecolor='black', lw=1.5, zorder=5)

# Formatting
ax.set_title("Density Peak Discovery within a Connected Component", fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel("\nFeature 1", fontsize=12)
ax.set_ylabel("\nFeature 2", fontsize=12)
ax.set_zlabel("\nLocal Density ($p_i$)", fontsize=12)

# Custom viewpoint for the best 3D angle
ax.view_init(elev=20, azim=-55)

# Custom Educational Legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=10, 
           label=r'1. Nodes elevated by Density ($p_i$)'),
    Line2D([0], [0], color='dimgray', lw=2, linestyle='--', 
           label=r'2. Uphill path to nearest denser node ($q_i$)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=12, 
           label=r'3. Macro-Cluster Centers ($h_i \geq \alpha \times h_{mean}$)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=13, frameon=True, shadow=True, borderpad=1)

plt.tight_layout()

# Save for LaTeX
plt.savefig("3D_density_peaks_clean.png", dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

# Add paths to load your actual project files
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

from src.densoinn import DenSOINN
from baseframework.datahandler import load_data

# ==========================================
# 1. Train DenSOINN on complex9
# ==========================================
print("Loading complex9 and training DenSOINN Online Phase...")
X_raw, y_raw = load_data('complex9', scale=True)

# Run the online phase to get the Neural Nodes
densoinn = DenSOINN(gamma=0.85, stream_speed=200, epsilon=0.5)
densoinn.learn(X_raw, np.arange(len(X_raw)))
summaries = densoinn.online_phase.get_summary()

# ==========================================
# 2. Step 1: Get Densities & Components
# ==========================================
# Calculate p_i exactly as your offline phase does
densities = {}
for node_id, data in summaries.items():
    m_i = data["weight"]
    w_i = data["centroid"]
    neighbors = data["neighbors"]
    if not neighbors:
        p_i = float(m_i)
    else:
        dists = [np.linalg.norm(w_i - summaries[nbr]["centroid"]) for nbr in neighbors]
        p_i = m_i / (1.0 + (np.mean(dists) ** 2)) # Intrinsic dim = 2
    densities[node_id] = p_i

# Run BFS to separate the islands
visited = set()
components = []
for node in summaries:
    if node not in visited:
        comp = []
        queue = [node]
        visited.add(node)
        while queue:
            curr = queue.pop(0)
            comp.append(curr)
            for nbr in summaries[curr]["neighbors"]:
                if nbr not in visited and nbr in summaries:
                    visited.add(nbr)
                    queue.append(nbr)
        components.append(comp)

# Pick the largest connected component (island) to visualize
largest_comp = max(components, key=len)
print(f"Visualizing a component with {len(largest_comp)} Neural Nodes.")

# Extract X (centroids) and p_i (densities)
X = np.array([summaries[nid]["centroid"] for nid in largest_comp])
p_arr = np.array([densities[nid] for nid in largest_comp])

# Scale p_arr purely for visual Z-axis proportions in the 3D plot
p_arr = (p_arr / np.max(p_arr)) * 100 

# Generate a smooth KDE surface for the mountain aesthetic
kde = gaussian_kde(X.T, bw_method=0.3)
x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
positions = np.vstack([xx.ravel(), yy.ravel()])
zz = np.reshape(kde(positions).T, xx.shape)
zz = (zz / np.max(zz)) * 100 # Match node height

# ==========================================
# 3. Simulate Step 2 Math (Find q_i and Peaks)
# ==========================================
q = {}
h = np.zeros(len(X))
sorted_indices = np.argsort(p_arr)[::-1]

# Global peak handling
global_peak = sorted_indices[0]
q[global_peak] = global_peak 
h[global_peak] = np.max(np.linalg.norm(X - X[global_peak], axis=1)) * p_arr[global_peak]

for k in range(1, len(sorted_indices)):
    i = sorted_indices[k]
    candidates = sorted_indices[:k] 
    dists = np.linalg.norm(X[candidates] - X[i], axis=1)
    min_idx = np.argmin(dists)
    
    q[i] = candidates[min_idx]
    h[i] = p_arr[i] * dists[min_idx]

# Apply the Alpha Threshold
h_mean = np.mean(h)
alpha = 2.0
centers = [i for i in range(len(X)) if h[i] >= alpha * h_mean]

# ==========================================
# 4. Plotting the 3D Academic Figure
# ==========================================
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Draw the Mountain Surface
ax.plot_surface(xx, yy, zz, cmap='Blues', alpha=0.3, edgecolor='none', zorder=0)

# Draw the Standard Neural Nodes
non_centers = [i for i in range(len(X)) if i not in centers]
ax.scatter(X[non_centers, 0], X[non_centers, 1], p_arr[non_centers], 
           c='black', s=50, depthshade=True, zorder=2)

# Draw the "Uphill" Paths (Distance to q_i)
for i in range(len(X)):
    if i != global_peak and i not in centers:
        target = q[i]
        ax.plot([X[i,0], X[target,0]], 
                [X[i,1], X[target,1]], 
                [p_arr[i], p_arr[target]], 
                color='dimgray', linestyle='--', linewidth=1.5, alpha=0.8)

# Highlight the Peaks (Red Nodes)
ax.scatter(X[centers, 0], X[centers, 1], p_arr[centers], 
           c='red', marker='o', s=120, edgecolor='black', lw=1.5, zorder=5)

# Formatting
ax.set_title("Density Peaks on actual complex9 Neural Nodes", fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel("\nFeature 1", fontsize=12)
ax.set_ylabel("\nFeature 2", fontsize=12)
ax.set_zlabel("\nLocal Density ($p_i$)", fontsize=12)

ax.view_init(elev=20, azim=-55)

# Custom Educational Legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=10, 
           label=r'1. Nodes elevated by Density ($p_i$)'),
    Line2D([0], [0], color='dimgray', lw=2, linestyle='--', 
           label=r'2. Uphill path to nearest denser node ($q_i$)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=12, 
           label=r'3. Macro-Cluster Centers ($h_i \geq \alpha h_{mean}$)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=13, frameon=True, shadow=True, borderpad=1)

plt.tight_layout()

# Save for LaTeX
# plt.savefig("3D_complex9_peaks.png", dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

# Add paths to load your actual project files
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

from src.densoinn import DenSOINN
from baseframework.datahandler import load_data

# ==========================================
# 1. Train DenSOINN on complex9
# ==========================================
print("Loading complex9 and training DenSOINN Online Phase...")
X_raw, y_raw = load_data('complex9', scale=True)

# Run the online phase to get the Neural Nodes
densoinn = DenSOINN(gamma=0.85, stream_speed=200, epsilon=0.5)
densoinn.learn(X_raw, np.arange(len(X_raw)))
summaries = densoinn.online_phase.get_summary()

# ==========================================
# 2. Step 1: Get Densities & Components
# ==========================================
densities = {}
for node_id, data in summaries.items():
    m_i = data["weight"]
    w_i = data["centroid"]
    neighbors = data["neighbors"]
    if not neighbors:
        p_i = float(m_i)
    else:
        dists = [np.linalg.norm(w_i - summaries[nbr]["centroid"]) for nbr in neighbors]
        p_i = m_i / (1.0 + (np.mean(dists) ** 2)) # Intrinsic dim = 2
    densities[node_id] = p_i

# Run BFS to separate the islands
visited = set()
components = []
for node in summaries:
    if node not in visited:
        comp = []
        queue = [node]
        visited.add(node)
        while queue:
            curr = queue.pop(0)
            comp.append(curr)
            for nbr in summaries[curr]["neighbors"]:
                if nbr not in visited and nbr in summaries:
                    visited.add(nbr)
                    queue.append(nbr)
        components.append(comp)

# Filter out tiny noise components (e.g., < 3 nodes) for a cleaner plot
valid_components = [c for c in components if len(c) >= 3]
print(f"Visualizing {len(valid_components)} distinct islands (components).")

# Extract all valid nodes to normalize Z-axis globally
all_valid_nodes = [nid for comp in valid_components for nid in comp]
X_all = np.array([summaries[nid]["centroid"] for nid in all_valid_nodes])
P_all = np.array([densities[nid] for nid in all_valid_nodes])

# Scale densities to 0-100 purely for visual Z-axis proportions
max_p = np.max(P_all)
for nid in densities:
    densities[nid] = (densities[nid] / max_p) * 100

# Generate a smooth KDE surface for the global mountain aesthetic
kde = gaussian_kde(X_all.T, bw_method=0.15)
x_min, x_max = X_all[:, 0].min() - 0.1, X_all[:, 0].max() + 0.1
y_min, y_max = X_all[:, 1].min() - 0.1, X_all[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
positions = np.vstack([xx.ravel(), yy.ravel()])
zz = np.reshape(kde(positions).T, xx.shape)
zz = (zz / np.max(zz)) * 100 # Match node height

# ==========================================
# 3. Plotting the 3D Academic Figure
# ==========================================
fig = plt.figure(figsize=(16, 11))
ax = fig.add_subplot(111, projection='3d')

# Draw the global Mountain Surface
ax.plot_surface(xx, yy, zz, cmap='Blues', alpha=0.25, edgecolor='none', zorder=0)

alpha_thresh = 6.0 # Slightly higher alpha since we have multiple components

# Loop through EVERY island and calculate its local peaks and slopes
for comp in valid_components:
    X_comp = np.array([summaries[nid]["centroid"] for nid in comp])
    p_comp = np.array([densities[nid] for nid in comp])
    
    q = {}
    h = np.zeros(len(comp))
    sorted_indices = np.argsort(p_comp)[::-1]

    # Global peak handling for this specific island
    global_peak = sorted_indices[0]
    q[global_peak] = global_peak 
    h[global_peak] = np.max(np.linalg.norm(X_comp - X_comp[global_peak], axis=1)) * p_comp[global_peak]

    # Find nearest higher-density node (q_i)
    for k in range(1, len(sorted_indices)):
        i = sorted_indices[k]
        candidates = sorted_indices[:k] 
        dists = np.linalg.norm(X_comp[candidates] - X_comp[i], axis=1)
        min_idx = np.argmin(dists)
        
        q[i] = candidates[min_idx]
        h[i] = p_comp[i] * dists[min_idx]

    # Apply the Alpha Threshold locally
    h_mean = np.mean(h)
    centers = [i for i in range(len(comp)) if h[i] >= alpha_thresh * h_mean]

    # Draw the Standard Neural Nodes
    non_centers = [i for i in range(len(comp)) if i not in centers]
    if len(non_centers) > 0:
        ax.scatter(X_comp[non_centers, 0], X_comp[non_centers, 1], p_comp[non_centers], 
                   c='black', s=25, depthshade=True, zorder=2)

    # Draw the "Uphill" Paths for this island
    for i in range(len(comp)):
        if i != global_peak and i not in centers:
            target = q[i]
            ax.plot([X_comp[i,0], X_comp[target,0]], 
                    [X_comp[i,1], X_comp[target,1]], 
                    [p_comp[i], p_comp[target]], 
                    color='dimgray', linestyle='--', linewidth=0.8, alpha=0.6)

    # Highlight the Peaks (Red Nodes)
    if len(centers) > 0:
        ax.scatter(X_comp[centers, 0], X_comp[centers, 1], p_comp[centers], 
                   c='red', marker='o', s=70, edgecolor='black', lw=1.0, zorder=5)

# Formatting
ax.set_title("Global Topology: Density Peaks Across All Components (complex9)", fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel("\nFeature 1", fontsize=12)
ax.set_ylabel("\nFeature 2", fontsize=12)
ax.set_zlabel("\nLocal Density ($p_i$)", fontsize=12)

# Elevated viewpoint to see over all the different islands
ax.view_init(elev=35, azim=-45)

# Custom Educational Legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=8, 
           label=r'1. Nodes elevated by Density ($p_i$)'),
    Line2D([0], [0], color='dimgray', lw=1.5, linestyle='--', 
           label=r'2. Uphill path to nearest denser node ($q_i$)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=10, 
           label=r'3. Macro-Cluster Centers ($h_i \geq \alpha h_{mean}$)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=13, frameon=True, shadow=True, borderpad=1)

plt.tight_layout()

# Save for LaTeX
# plt.savefig("3D_global_density_peaks.png", dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Add paths to load your actual project files
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

from src.densoinn import DenSOINN
from baseframework.datahandler import load_data

# ==========================================
# 1. Train DenSOINN on complex9
# ==========================================
print("Loading complex9 and training DenSOINN Online Phase...")
X_raw, y_raw = load_data('complex9', scale=True)

# Run the online phase to get the Neural Nodes
densoinn = DenSOINN(gamma=0.25, stream_speed=200, epsilon=0.5)
densoinn.learn(X_raw, np.arange(len(X_raw)))
summaries = densoinn.online_phase.get_summary()

# ==========================================
# 2. Step 1: Get Densities & Components
# ==========================================
densities = {}
for node_id, data in summaries.items():
    m_i = data["weight"]
    w_i = data["centroid"]
    neighbors = data["neighbors"]
    if not neighbors:
        p_i = float(m_i)
    else:
        dists = [np.linalg.norm(w_i - summaries[nbr]["centroid"]) for nbr in neighbors]
        p_i = m_i / (1.0 + (np.mean(dists) ** 2)) # Intrinsic dim = 2
    densities[node_id] = p_i

# Run BFS to separate the islands
visited = set()
components = []
for node in summaries:
    if node not in visited:
        comp = []
        queue = [node]
        visited.add(node)
        while queue:
            curr = queue.pop(0)
            comp.append(curr)
            for nbr in summaries[curr]["neighbors"]:
                if nbr not in visited and nbr in summaries:
                    visited.add(nbr)
                    queue.append(nbr)
        components.append(comp)

# Filter out tiny noise components
valid_components = [c for c in components if len(c) >= 3]
print(f"Visualizing {len(valid_components)} distinct islands (components).")

# Extract all valid nodes
all_valid_nodes = [nid for comp in valid_components for nid in comp]
X_all = np.array([summaries[nid]["centroid"] for nid in all_valid_nodes])
P_all = np.array([densities[nid] for nid in all_valid_nodes])

# Scale densities to 0-100 purely for visual calculations
P_all_scaled = (P_all / np.max(P_all)) * 100

# ==========================================
# NEW: Build Custom Terrain weighted by EXACT p_i
# ==========================================
x_min, x_max = X_all[:, 0].min() - 0.1, X_all[:, 0].max() + 0.1
y_min, y_max = X_all[:, 1].min() - 0.1, X_all[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))

# Define the "width" of the mountains
sigma = 0.035 
zz_raw = np.zeros_like(xx)

# For every node, add a mountain whose height is driven by its algorithm density p_i
for idx, (x_node, y_node) in enumerate(X_all):
    dist_sq = (xx - x_node)**2 + (yy - y_node)**2
    zz_raw += P_all_scaled[idx] * np.exp(-dist_sq / (2 * sigma**2))

# Normalize surface height
terrain_max = np.max(zz_raw)
zz = (zz_raw / terrain_max) * 100 

# ==========================================
# 3. Simulate Step 2 Math (Find q_i and Peaks)
# ==========================================
fig = plt.figure(figsize=(16, 11))
ax = fig.add_subplot(111, projection='3d')

# Draw the custom Mountain Surface
ax.plot_surface(xx, yy, zz, cmap='Blues', alpha=0.5, edgecolor='none', zorder=0)

alpha_thresh = 3.0 # Threshold for peak selection

# Loop through EVERY island and find its local peaks
for comp in valid_components:
    X_comp = np.array([summaries[nid]["centroid"] for nid in comp])
    p_comp = np.array([densities[nid] for nid in comp])
    
    q = {}
    h = np.zeros(len(comp))
    sorted_indices = np.argsort(p_comp)[::-1]

    # Global peak handling for this specific island
    global_peak = sorted_indices[0]
    q[global_peak] = global_peak 
    h[global_peak] = np.max(np.linalg.norm(X_comp - X_comp[global_peak], axis=1)) * p_comp[global_peak]

    # Find nearest higher-density node (q_i)
    for k in range(1, len(sorted_indices)):
        i = sorted_indices[k]
        candidates = sorted_indices[:k] 
        dists = np.linalg.norm(X_comp[candidates] - X_comp[i], axis=1)
        min_idx = np.argmin(dists)
        
        q[i] = candidates[min_idx]
        h[i] = p_comp[i] * dists[min_idx]

    # Apply the Alpha Threshold locally
    h_mean = np.mean(h)
    centers = [i for i in range(len(comp)) if h[i] >= alpha_thresh * h_mean]

    # Highlight ONLY the Peaks (Red Nodes)
    if len(centers) > 0:
        x_centers = X_comp[centers, 0]
        y_centers = X_comp[centers, 1]
        
        # NEW: Calculate exact Z-height on the custom terrain so they sit perfectly flush
        z_centers = np.zeros(len(centers))
        for c_idx in range(len(centers)):
            dist_sq = (X_all[:,0] - x_centers[c_idx])**2 + (X_all[:,1] - y_centers[c_idx])**2
            raw_height = np.sum(P_all_scaled * np.exp(-dist_sq / (2 * sigma**2)))
            z_centers[c_idx] = (raw_height / terrain_max) * 100
        
        # Plot the Red Centers
        ax.scatter(x_centers, y_centers, z_centers, 
                   c='red', marker='o', s=160, edgecolor='black', lw=1.5, zorder=5)

# Formatting
ax.set_title("Global Topology: Density Peaks Across All Islands (Complex9)", fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel("\nFeature 1", fontsize=12)
ax.set_ylabel("\nFeature 2", fontsize=12)
ax.set_zlabel("\nLocal Density ($p_i$)", fontsize=12)

# Prevent the "squashed" 3D cube look
ax.set_box_aspect((1, 1, 0.75))

# Elevated viewpoint to see over all the different islands
ax.view_init(elev=40, azim=-55)

# Custom Educational Legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=12, 
           label=r'Macro-Cluster Centers')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=14, frameon=True, shadow=True, borderpad=1)

plt.tight_layout()

# Save for LaTeX
plt.savefig("3D_flawless_density_peaks.png", dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [20]:
import numpy as np
from baseframework.datahandler import load_data

datasets = ['complex9', 'rbf3', 'har', 'spiral', 'stream_densired']

print(f"{'Dataset':<16} | {'Max Range Ratio':<16} | {'Max Variance Ratio'}")
print("-" * 55)

for ds in datasets:
    # Load RAW data
    X, _ = load_data(ds, scale=False)
    
    # 1. Calculate the Range (Max - Min) of each feature
    ranges = np.ptp(X, axis=0) # ptp = peak-to-peak (max - min)
    
    # Avoid division by zero for constant features
    ranges = ranges[ranges > 0] 
    range_ratio = np.max(ranges) / np.min(ranges) if len(ranges) > 0 else 1.0
    
    # 2. Calculate the Variance of each feature
    variances = np.var(X, axis=0)
    variances = variances[variances > 0]
    var_ratio = np.max(variances) / np.min(variances) if len(variances) > 0 else 1.0
    
    print(f"{ds:<16} | {range_ratio:<16.2f} | {var_ratio:.2f}")

Dataset          | Max Range Ratio  | Max Variance Ratio
-------------------------------------------------------
complex9         | 1.59             | 2.83
rbf3             | 1.00             | 1.31
har              | 1.00             | 407.49
spiral           | 1.11             | 1.02
stream_densired  | 1.12             | 1.02


In [22]:
import pandas as pd
import numpy as np
import os

def prepare_kdd99_local():
    print("Reading local KDD99 CSV file...")
    
    # Read the local CSV file
    df = pd.read_csv("./data/kddcup99_csv.csv")

    # The last column is the label
    y_raw = df['label'].values

    # Convert string labels (like 'normal', 'smurf', etc.) to integer IDs
    unique_labels = np.unique(y_raw)
    label_map = {val: i for i, val in enumerate(unique_labels)}
    y = np.array([label_map[val] for val in y_raw])

    # Drop the label column to isolate features
    X_df = df.drop(columns=['label'])

    # One-hot encode the categorical string features 
    # pandas get_dummies automatically finds the string/object columns (protocol_type, service, flag)
    X_df = pd.get_dummies(X_df)

    # Convert to a pure numeric numpy array
    X = X_df.values.astype(np.float64)

    # Save to your data folder
    np.save("./data/kdd99_10pct_data.npy", X)
    np.save("./data/kdd99_10pct_label.npy", y)

    print(f"\nSuccess! Saved to ./data/")
    print(f"X shape: {X.shape} (Features increased due to one-hot encoding)")
    print(f"y shape: {y.shape} (Total patterns)")
    
    # Calculate Range Ratio to show the extreme scale disparity
    ranges = np.ptp(X, axis=0)
    ranges = ranges[ranges > 0] 
    range_ratio = np.max(ranges) / np.min(ranges) if len(ranges) > 0 else 1.0
    
    # Calculate Variance Ratio
    variances = np.var(X, axis=0)
    variances = variances[variances > 0]
    var_ratio = np.max(variances) / np.min(variances) if len(variances) > 0 else 1.0
    
    print(f"\n--> KDD99 Max Range Ratio: {range_ratio:,.2f}")
    print(f"--> KDD99 Max Var Ratio:   {var_ratio:,.2f}")
    print("    (This massive scale disparity is exactly what DenSOINN fixes!)")

if __name__ == "__main__":
    prepare_kdd99_local()

Reading local KDD99 CSV file...

Success! Saved to ./data/
X shape: (494020, 118) (Features increased due to one-hot encoding)
y shape: (494020,) (Total patterns)

--> KDD99 Max Range Ratio: 693,375,640.00
--> KDD99 Max Var Ratio:   482,448,565,600,411,200.00
    (This massive scale disparity is exactly what DenSOINN fixes!)


In [23]:
import pandas as pd
import numpy as np

def prepare_kdd99_mini():
    print("Reading local KDD99 CSV file...")
    df = pd.read_csv("./data/kddcup99_csv.csv")

    # OPTIONAL: Take a random 10% sample of this file for faster execution
    # (This reduces it from 494k rows to ~49k rows)
    df = df.sample(frac=0.10, random_state=42).reset_index(drop=True)
    print(f"Sampled down to {len(df)} rows for faster processing.")

    # The last column is the label
    y_raw = df['label'].values

    unique_labels = np.unique(y_raw)
    label_map = {val: i for i, val in enumerate(unique_labels)}
    y = np.array([label_map[val] for val in y_raw])

    # Drop the label column to isolate features
    X_df = df.drop(columns=['label'])

    # One-hot encode
    X_df = pd.get_dummies(X_df)
    X = X_df.values.astype(np.float64)

    # Save to your data folder under a NEW name so you don't overwrite the big one
    np.save("./data/kdd99_mini_data.npy", X)
    np.save("./data/kdd99_mini_label.npy", y)

    print(f"\nSuccess! Saved to ./data/kdd99_mini_...")
    print(f"X shape: {X.shape}")
    
    ranges = np.ptp(X, axis=0)
    ranges = ranges[ranges > 0] 
    range_ratio = np.max(ranges) / np.min(ranges) if len(ranges) > 0 else 1.0
    print(f"\n--> Mini-KDD99 Max Range Ratio: {range_ratio:,.2f}")

if __name__ == "__main__":
    prepare_kdd99_mini()

Reading local KDD99 CSV file...
Sampled down to 49402 rows for faster processing.

Success! Saved to ./data/kdd99_mini_...
X shape: (49402, 115)

--> Mini-KDD99 Max Range Ratio: 5,135,678.00


In [27]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import MinMaxScaler

# Load the KDD99 data (use the mini 10% sample if you made it)
X = np.load("./data/kdd99_mini_data.npy")
y = np.load("./data/kdd99_mini_label.npy")

print("Running K-Means on RAW data...")
# We use k=23 because there are 23 true classes in KDD99 10%
kmeans_raw = MiniBatchKMeans(n_clusters=23, random_state=42, n_init="auto")
labels_raw = kmeans_raw.fit_predict(X)
print(f"RAW NMI: {normalized_mutual_info_score(y, labels_raw):.4f}")
print(f"RAW ARI: {adjusted_rand_score(y, labels_raw):.4f}")

print("\nRunning K-Means on SCALED data...")
X_scaled = MinMaxScaler().fit_transform(X)
kmeans_scaled = MiniBatchKMeans(n_clusters=23, random_state=42, n_init="auto")
labels_scaled = kmeans_scaled.fit_predict(X_scaled)
print(f"SCALED NMI: {normalized_mutual_info_score(y, labels_scaled):.4f}")
print(f"SCALED ARI: {adjusted_rand_score(y, labels_scaled):.4f}")

Running K-Means on RAW data...
RAW NMI: 0.6034
RAW ARI: 0.5002

Running K-Means on SCALED data...
SCALED NMI: 0.6140
SCALED ARI: 0.6030


In [28]:
import sys
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import MinMaxScaler

# Ensure Python can find your baseframework imports
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

# Import your custom stream metrics
from baseframework.metrics.cmm import get_cmm
from baseframework.metrics.TSindex import tempsil

def main():
    print("Loading KDD99 Mini dataset...")
    X = np.load("./data/kdd99_mini_data.npy")
    y = np.load("./data/kdd99_mini_label.npy")

    # Arrays required by your metric functions
    w = np.ones(len(X))
    timestamps = np.arange(len(X))

    # ==========================================
    # 1. RAW DATA (The Failure Case)
    # ==========================================
    print("\n--- Running K-Means on RAW data ---")
    kmeans_raw = MiniBatchKMeans(n_clusters=23, random_state=42, n_init="auto")
    labels_raw = kmeans_raw.fit_predict(X)

    print("Calculating metrics (CMM and TS might take a minute)...")
    nmi_raw = normalized_mutual_info_score(y, labels_raw)
    ari_raw = adjusted_rand_score(y, labels_raw)
    
    # We pass the raw X to CMM and TS
    cmm_raw = get_cmm(X, y, labels_raw, w, k=5)
    _, _, ts_raw = tempsil(timestamps, X, labels_raw)

    print(f"RAW NMI: {nmi_raw:.3f}")
    print(f"RAW ARI: {ari_raw:.3f}")
    print(f"RAW CMM: {cmm_raw:.3f}")
    print(f"RAW TS:  {ts_raw:.3f}")

    # ==========================================
    # 2. SCALED DATA (The Baseline)
    # ==========================================
    print("\n--- Running K-Means on SCALED data ---")
    X_scaled = MinMaxScaler().fit_transform(X)
    
    kmeans_scaled = MiniBatchKMeans(n_clusters=23, random_state=42, n_init="auto")
    labels_scaled = kmeans_scaled.fit_predict(X_scaled)

    print("Calculating metrics (CMM and TS might take a minute)...")
    nmi_scaled = normalized_mutual_info_score(y, labels_scaled)
    ari_scaled = adjusted_rand_score(y, labels_scaled)
    
    # We pass the scaled X to CMM and TS to match the space it clustered in
    cmm_scaled = get_cmm(X_scaled, y, labels_scaled, w, k=5)
    _, _, ts_scaled = tempsil(timestamps, X_scaled, labels_scaled)

    print(f"SCALED NMI: {nmi_scaled:.3f}")
    print(f"SCALED ARI: {ari_scaled:.3f}")
    print(f"SCALED CMM: {cmm_scaled:.3f}")
    print(f"SCALED TS:  {ts_scaled:.3f}")

if __name__ == "__main__":
    main()

Loading KDD99 Mini dataset...

--- Running K-Means on RAW data ---
Calculating metrics (CMM and TS might take a minute)...
RAW NMI: 0.603
RAW ARI: 0.500
RAW CMM: 0.958
RAW TS:  0.679

--- Running K-Means on SCALED data ---
Calculating metrics (CMM and TS might take a minute)...
SCALED NMI: 0.614
SCALED ARI: 0.603
SCALED CMM: 0.918
SCALED TS:  0.441


In [ ]:
import pandas as pd
import numpy as np
import os
import kagglehub

def prepare_covtype():
    print("Downloading Forest Cover Type dataset via kagglehub...")
    # Download latest version from Kaggle
    path = kagglehub.dataset_download("uciml/forest-cover-type-dataset")
    csv_path = os.path.join(path, "covtype.csv")
    
    print(f"Loading full dataset from: {csv_path}")
    df = pd.read_csv(csv_path)

    # The last column is the label (Cover_Type, typically 1-7)
    X_full = df.iloc[:, :-1].values.astype(np.float64)
    y_full = df.iloc[:, -1].values
    
    # Standardize labels to 0-indexed (0 to 6)
    unique_labels = np.unique(y_full)
    label_map = {val: i for i, val in enumerate(unique_labels)}
    y_full = np.array([label_map[val] for val in y_full])

    print("\n--- ANALYZING FULL DATASET FOR SLIDE 1 ---")
    print(f"Total shape: {X_full.shape}")
    
    # Calculate Range Ratio
    ranges = np.ptp(X_full, axis=0)
    ranges = ranges[ranges > 0] 
    range_ratio = np.max(ranges) / np.min(ranges) if len(ranges) > 0 else 1.0
    
    # Calculate Variance Ratio
    variances = np.var(X_full, axis=0)
    variances = variances[variances > 0]
    var_ratio = np.max(variances) / np.min(variances) if len(variances) > 0 else 1.0
    
    print(f"--> Covertype Max Range Ratio: {range_ratio:,.2f}")
    print(f"--> Covertype Max Var Ratio:   {var_ratio:,.2f}")
    
    # ---------------------------------------------------------
    # Create the 10k Mini Subset for testing
    # ---------------------------------------------------------
    print("\n--- CREATING 10k MINI SUBSET ---")
    df_mini = df.sample(n=10000, random_state=42).reset_index(drop=True)
    
    X_mini = df_mini.iloc[:, :-1].values.astype(np.float64)
    y_mini_raw = df_mini.iloc[:, -1].values
    y_mini = np.array([label_map[val] for val in y_mini_raw])

    os.makedirs("./data", exist_ok=True)
    np.save("./data/covtype_mini_data.npy", X_mini)
    np.save("./data/covtype_mini_label.npy", y_mini)

    print("Success! Saved 10k subset to ./data/covtype_mini_...")

if __name__ == "__main__":
    prepare_covtype()

100%|██████████| 11.2M/11.2M [00:01<00:00, 10.9MB/s]

Extracting files...


Loading full dataset from: /Users/felix/.cache/kagglehub/datasets/uciml/forest-cover-type-dataset/versions/1/covtype.csv

--- ANALYZING FULL DATASET FOR SLIDE 1 ---
Total shape: (581012, 54)
--> Covertype Max Range Ratio: 7,173.00
--> Covertype Max Var Ratio:   470,868,416,076.18

--- CREATING 10k MINI SUBSET ---
Success! Saved 10k subset to ./data/covtype_mini_...


In [32]:
import sys
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import MinMaxScaler

# Ensure Python can find your baseframework imports
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

# Import your custom stream metrics
try:
    from baseframework.metrics.cmm import get_cmm
    from baseframework.metrics.TSindex import tempsil
except ImportError:
    print("Warning: Could not import custom metrics!")

def main():
    print("Loading CovType Mini dataset...")
    try:
        X = np.load("./data/covtype_mini_data.npy")
        y = np.load("./data/covtype_mini_label.npy")
    except FileNotFoundError:
        print("Error: Could not find covtype_mini data. Did you run setup_covtype.py?")
        return

    # Arrays required by your metric functions
    w = np.ones(len(X))
    timestamps = np.arange(len(X))

    # ==========================================
    # 1. RAW DATA (The Failure Case)
    # ==========================================
    print("\n--- Running K-Means on RAW data ---")
    kmeans_raw = MiniBatchKMeans(n_clusters=7, random_state=42, n_init="auto")
    labels_raw = kmeans_raw.fit_predict(X)

    print("Calculating metrics (CMM and TS might take a minute)...")
    nmi_raw = normalized_mutual_info_score(y, labels_raw)
    ari_raw = adjusted_rand_score(y, labels_raw)
    
    # We pass the raw X to CMM and TS
    cmm_raw = get_cmm(X, y, labels_raw, w, k=5)
    _, _, ts_raw = tempsil(timestamps, X, labels_raw)

    print(f"RAW NMI: {nmi_raw:.3f}")
    print(f"RAW ARI: {ari_raw:.3f}")
    print(f"RAW CMM: {cmm_raw:.3f}")
    print(f"RAW TS:  {ts_raw:.3f}")

    # ==========================================
    # 2. SCALED DATA (The Baseline)
    # ==========================================
    print("\n--- Running K-Means on SCALED data ---")
    X_scaled = MinMaxScaler().fit_transform(X)
    
    kmeans_scaled = MiniBatchKMeans(n_clusters=7, random_state=42, n_init="auto")
    labels_scaled = kmeans_scaled.fit_predict(X_scaled)

    print("Calculating metrics (CMM and TS might take a minute)...")
    nmi_scaled = normalized_mutual_info_score(y, labels_scaled)
    ari_scaled = adjusted_rand_score(y, labels_scaled)
    
    # We pass the scaled X to CMM and TS to match the space it clustered in
    cmm_scaled = get_cmm(X_scaled, y, labels_scaled, w, k=5)
    _, _, ts_scaled = tempsil(timestamps, X_scaled, labels_scaled)

    print(f"SCALED NMI: {nmi_scaled:.3f}")
    print(f"SCALED ARI: {ari_scaled:.3f}")
    print(f"SCALED CMM: {cmm_scaled:.3f}")
    print(f"SCALED TS:  {ts_scaled:.3f}")

if __name__ == "__main__":
    main()

Loading CovType Mini dataset...

--- Running K-Means on RAW data ---
Calculating metrics (CMM and TS might take a minute)...
RAW NMI: 0.078
RAW ARI: 0.012
RAW CMM: 0.869
RAW TS:  0.379

--- Running K-Means on SCALED data ---
Calculating metrics (CMM and TS might take a minute)...
SCALED NMI: 0.162
SCALED ARI: 0.085
SCALED CMM: 0.879
SCALED TS:  0.272


In [33]:
def prepare_covtype():
    print("Downloading Forest Cover Type dataset via kagglehub...")
    # Download latest version from Kaggle
    path = kagglehub.dataset_download("uciml/forest-cover-type-dataset")
    csv_path = os.path.join(path, "covtype.csv")
    
    print(f"Loading full dataset from: {csv_path}")
    df = pd.read_csv(csv_path)

    # The last column is the label (Cover_Type, typically 1-7)
    X_full = df.iloc[:, :-1].values.astype(np.float64)
    y_full = df.iloc[:, -1].values
    
    # Standardize labels to 0-indexed (0 to 6)
    unique_labels = np.unique(y_full)
    label_map = {val: i for i, val in enumerate(unique_labels)}
    y_full = np.array([label_map[val] for val in y_full])

    print("\n--- ANALYZING FULL DATASET FOR SLIDE 1 ---")
    print(f"Total shape: {X_full.shape}")
    
    # Calculate Range Ratio
    ranges = np.ptp(X_full, axis=0)
    ranges = ranges[ranges > 0] 
    range_ratio = np.max(ranges) / np.min(ranges) if len(ranges) > 0 else 1.0
    
    # Calculate Variance Ratio
    variances = np.var(X_full, axis=0)
    variances = variances[variances > 0]
    var_ratio = np.max(variances) / np.min(variances) if len(variances) > 0 else 1.0
    
    print(f"--> Covertype Max Range Ratio: {range_ratio:,.2f}")
    print(f"--> Covertype Max Var Ratio:   {var_ratio:,.2f}")
    
    # ---------------------------------------------------------
    # Create the 10k Mini Subset for testing
    # ---------------------------------------------------------
    print("\n--- CREATING 50k MINI SUBSET ---")
    df_mini = df.sample(n=50000, random_state=42).reset_index(drop=True)
    
    X_mini = df_mini.iloc[:, :-1].values.astype(np.float64)
    y_mini_raw = df_mini.iloc[:, -1].values
    y_mini = np.array([label_map[val] for val in y_mini_raw])

    os.makedirs("./data", exist_ok=True)
    np.save("./data/covtype_50k_data.npy", X_mini)
    np.save("./data/covtype_50k_label.npy", y_mini)

    print("Success! Saved 50k subset to ./data/covtype_50k_...")

if __name__ == "__main__":
    prepare_covtype()

Loading full dataset from: /Users/felix/.cache/kagglehub/datasets/uciml/forest-cover-type-dataset/versions/1/covtype.csv

--- ANALYZING FULL DATASET FOR SLIDE 1 ---
Total shape: (581012, 54)
--> Covertype Max Range Ratio: 7,173.00
--> Covertype Max Var Ratio:   470,868,416,076.18

--- CREATING 50k MINI SUBSET ---
Success! Saved 50k subset to ./data/covtype_50k_...
